# Updating a PMO with new metadata 

Creating a PMO from the data downloaded from the SRA associated with the following study and updating with meta data:

Furstenau, T. N., Whealy, R., Timm, S., Roberts, A., Maltinsky, S., Wells, S. J., Drake, K., Ross, A., Bolduc, C., Pearson, T., & Fofanov, V. Y. (2025). *High-throughput targeted amplicon screening tool for characterizing intrahost diversity in* Staphylococcus aureus *directly from sample*. Microbial Genomics, 11(6). https://doi.org/10.1099/mgen.0.001427

This will cover how to update specimen meta after building a minimum PMO

In [7]:
import pandas as pd
from pmotools.pmo_builder.panel_information_to_pmo import panel_info_table_to_pmo, merge_panel_info_dicts
from pmotools.pmo_builder.metatable_to_pmo import library_sample_info_table_to_pmo, specimen_info_table_to_pmo
from pmotools.pmo_builder.mhap_table_to_pmo import (
    mhap_table_to_pmo, 
    create_minimum_library_specimen_dict_from_mhap_table)
from pmotools.pmo_builder.merge_to_pmo import merge_to_pmo
from pmotools.pmo_engine.pmo_writer import * 
import numpy as np
from pmotools.pmo_builder import panel_information_to_pmo

## Read in data

The minimum amount of information needed to create a PMO is the microhaplotype data and information on the panel used (at a minimum, the target's primers). PMO has the greatest advantage by combining with sample metadata as well as the metadata associated with the sequencing and the bioinformatics used, all of which can be added to the first minimum PMO.  

* allele_data.tsv.gz - this file has the results of microhaplotype called data
* Furstenau2025_primers.tsv - this has the primers used in the experiment
* sra_info_table.tsv - this has the SRA/ENA meta information

In [8]:
mhap_info_df = pd.read_csv("allele_data.tsv.gz", sep='\t')
mhap_info_df.head()

,s_Sample,p_name,h_popUID,p_TotalPopulationSampCnt,h_AATyped,h_PopFrac,h_SumOfAllFracs,h_AvgFracFoundAt,h_ReadFrac,h_SampCnt,...,R1_totalCntExcluded,R1_totalFracExcluded,R1_clusterCntChiExcluded,R1_cntChiExcluded,R1_fracChiExcluded,R1_MapFrac,R1_ReadCnt,R1_ClusCnt,R1_totalReadCnt,c_bestExpected
0,SRR31969808,SA_1021483,SA_1021483.0,23,NaN,0.521739,12.0,1.0,0.508696,12,...,0,0.0,0,0,0.0,1.0,529,1,529,NaN
1,SRR31969809,SA_1021483,SA_1021483.0,23,NaN,0.521739,12.0,1.0,0.508696,12,...,0,0.0,0,0,0.0,1.0,452,1,452,NaN
2,SRR31969810,SA_1021483,SA_1021483.2,23,NaN,0.130435,3.0,1.0,0.148505,3,...,0,0.0,0,0,0.0,1.0,290,1,290,NaN
3,SRR31969811,SA_1021483,SA_1021483.1,23,NaN,0.217391,5.0,1.0,0.228397,5,...,0,0.0,0,0,0.0,1.0,408,1,408,NaN
4,SRR31969813,SA_1021483,SA_1021483.0,23,NaN,0.521739,12.0,1.0,0.508696,12,...,0,0.0,0,0,0.0,1.0,137,1,137,NaN


In [9]:
primers = pd.read_csv("Furstenau2025_primers.tsv", sep='\t')
primers.head()

,target,forward,reverse
0,SA_131432,GTCCAGGTAGCATGATT,TGTCATACCAGTTAGGAATCACA
1,SA_166442,AATTAAGTAAGCTCCAATGCGTT,TAGTTCGCTCTCCCCTTA
2,SA_219791,TCCAATATCCTGGCGTGA,TTCACAACCATTACCAAG
3,SA_303281,TAACGATGCGACAGGTACAG,ATGATGATGCTATGCGT
4,SA_433466,AGGTCTCACGACATCATT,CATAATACCTGCGCCATCA


## Panel Information
Panel information contains the information about the targets used to amplify the samples. Create panel information using the primers and a panel name using `panel_info_table_to_pmo`

In [10]:
pmo_panel_and_target_info = panel_info_table_to_pmo(primers, 
                                                    panel_name = "staph_aureus_Furstenau2025",
                                                    target_name_col = "target",
                                                    forward_primers_seq_col = "forward", 
                                                    reverse_primers_seq_col = "reverse")

## Microhaplotype Info

Creating microhaplotype data from the allele calls 

In [11]:
pmo_mhaps = mhap_table_to_pmo(
                       microhaplotype_table=mhap_info_df, 
                       library_sample_name_col='s_Sample',
                       target_name_col='p_name',
                       seq_col='h_Consensus',
                       reads_col='c_ReadCnt')

## Specimen and Library Sample Names

Will create library_sample_info and specimen_info by renaming the specimen to the sample name in the SRA data table

### Adding a key to set a specimen_name for library_sample_name 

A key can be provided to set a specimen_name for the library_sample_name. 

Read in the SRA information which can be used to create a key to change the specimen_names

In [12]:
sra_info = pd.read_csv("sra_info_table.tsv", sep = '\t')
# create a dictionary key
lib_to_spec_key = sra_info.set_index('run_accession')['sample_alias'].to_dict()

# supply key when building library_sample_info and specimen_info 
library_sample_and_spec_renamed_infos = create_minimum_library_specimen_dict_from_mhap_table(
    pmo_mhaps["detected_microhaplotypes"], 
    panel_name = "staph_aureus_Furstenau2025", 
    library_sample_specimen_key = lib_to_spec_key)

# now build with renamed 
staph_aureus_pmo_renamed = merge_to_pmo(
    specimen_info = library_sample_and_spec_renamed_infos["specimen_info"],
    library_sample_info = library_sample_and_spec_renamed_infos["library_sample_info"],
    panel_and_target_info = pmo_panel_and_target_info,
    mhap_info = pmo_mhaps
)

lib_to_spec_renamed_df = PMOExporter.list_library_sample_names_per_specimen_name(staph_aureus_pmo_renamed)
lib_to_spec_renamed_df.head()

,specimen_name,library_sample_name,library_sample_count
0,85b498-Wk16-Nasal,SRR30825770,1
1,85b498-Wk28-Nasal,SRR30825771,1
2,85b498-Wk12-Nasal,SRR30825772,1
3,85b498-Wk20-Nasal,SRR30825773,1
4,85b498-Wk14-Nasal,SRR30825774,1


### Adding in specimen meta information 

Right now the specimen_info is simply the specimen_name. Additional info can be joined to this barebones 


In [13]:
staph_aureus_pmo_renamed["specimen_info"][0]

{'specimen_name': '85b498-Wk16-Nasal'}

First use `specimen_info_table_to_pmo` and then merge into the specimen_info already present. There are several columns in the SRA/ENA metadata but for now we use as example the geographic location `country` and the date of collection `collection_date`.



In [14]:
sra_meta_of_interest = sra_info[['sample_alias', 'country', 'collection_date']].drop_duplicates()
sra_meta_of_interest.head()

,sample_alias,country,collection_date
0,2b2068n1,USA: Arizona,2019
2,2a4023n1,USA: Arizona,2019
3,2b4022n1,USA: Arizona,2019
5,2b3034n1,USA: Arizona,2019
9,2b2048n1,USA: Arizona,2019


We will want a field for country only and then state so will create new columns by split on ":" 

In [15]:
sra_meta_of_interest[['country_only', 'state']] = sra_meta_of_interest['country'].str.split(':', expand=True)
sra_meta_of_interest.head()

,sample_alias,country,collection_date,country_only,state
0,2b2068n1,USA: Arizona,2019,USA,Arizona
2,2a4023n1,USA: Arizona,2019,USA,Arizona
3,2b4022n1,USA: Arizona,2019,USA,Arizona
5,2b3034n1,USA: Arizona,2019,USA,Arizona
9,2b2048n1,USA: Arizona,2019,USA,Arizona


Now build the specimen meta 

In [16]:
pmo_spec_info = specimen_info_table_to_pmo(
                            sra_meta_of_interest, 
                            specimen_name_col='sample_alias',
                            collection_date_col='collection_date',
                            collection_country_col='country_only',
                            geo_admin1_col='state',
                           )

Now join this into the specimen_info using `PMOUpdater.merge_dicts_by_key`

In [17]:
from pmotools.pmo_builder.pmo_updater import PMOUpdater

staph_aureus_pmo_renamed["specimen_info"] = PMOUpdater.merge_dicts_by_key(
    staph_aureus_pmo_renamed["specimen_info"],
    pmo_spec_info, 
    key_field="specimen_name")

Now the specimen_info has meta data 

In [18]:
staph_aureus_pmo_renamed["specimen_info"][0]

{'specimen_name': '85b498-Wk16-Nasal',
 'collection_date': '2022-02-07',
 'collection_country': 'USA',
 'geo_admin1': ' Phoenix'}

Now let's write and validate the pmo with meta 

In [20]:
pmowriter = PMOWriter()
pmowriter.write_out_pmo(staph_aureus_pmo_renamed, "minimum_Furstenau2025_PMO_with_spec_meta.json.gz", overwrite=True)

In [21]:
!pmotools-python validate_pmo --pmo minimum_Furstenau2025_PMO_with_spec_meta.json.gz --jsonschema_version 1.1.0